# Olist Brazilian E-commerce — Phase 2: Data Cleaning & Preparation

**Business question:** Where is revenue coming from, which customers are most valuable, and what is hurting customer satisfaction?

This notebook performs data-quality inspection, key validation, datetime conversion, numeric-value checks, and conservative cleaning.

## Important rules
- Keep the original CSV files unchanged in `data/raw/`.
- Do not delete rows just to remove NULL values.
- Do not treat repeated foreign keys as duplicate records.
- Investigate anomalies before changing values.
- Preserve source-data issues in the cleaned datasets unless there is a justified transformation.
- The notebook records validation results so the cleaning process is reproducible.

**Expected folder structure**
```text
project/
├── data/
│   ├── raw/
│   └── processed/
└── notebooks/
    └── 02_data_cleaning.ipynb
```


## 1. Import libraries and configure paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data directory:", DATA_DIR.resolve())
print("Processed output directory:", OUTPUT_DIR.resolve())

Raw data directory: D:\Projects\olist-ecommerce-analysis\data\raw
Processed output directory: D:\Projects\olist-ecommerce-analysis\data\processed


## 2. Load the seven raw datasets

In [2]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers
}

for name, df in datasets.items():
    print(f"{name:15} rows={len(df):>7,} columns={df.shape[1]:>2}")

customers       rows= 99,441 columns= 5
orders          rows= 99,441 columns= 8
order_items     rows=112,650 columns= 7
payments        rows=103,886 columns= 5
reviews         rows= 99,224 columns= 7
products        rows= 32,951 columns= 9
sellers         rows=  3,095 columns= 4


## 3. Initial data-quality report

The report checks table size, completely duplicated rows, missing values, missing percentages, and unique values.

`df.duplicated()` checks whether every column in a row is identical. It does **not** check whether an ID column is unique.


In [3]:
def data_quality_report(df, table_name):
    report = pd.DataFrame({
        "column": df.columns,
        "data_type": df.dtypes.astype(str).values,
        "missing_values": df.isna().sum().values,
        "missing_percentage": (df.isna().mean().values * 100).round(2),
        "unique_values": df.nunique(dropna=True).values
    })

    print(f"\\nData quality report: {table_name}")
    print("Rows:", len(df))
    print("Columns:", df.shape[1])
    print("Completely duplicated rows:", df.duplicated().sum())
    display(report)

    return report

quality_reports = {
    name: data_quality_report(df, name)
    for name, df in datasets.items()
}

\nData quality report: customers
Rows: 99441
Columns: 5
Completely duplicated rows: 0


,column,data_type,missing_values,missing_percentage,unique_values
0,customer_id,str,0,0.0,99441
1,customer_unique_id,str,0,0.0,96096
2,customer_zip_code_prefix,int64,0,0.0,14994
3,customer_city,str,0,0.0,4119
4,customer_state,str,0,0.0,27


\nData quality report: orders
Rows: 99441
Columns: 8
Completely duplicated rows: 0


,column,data_type,missing_values,missing_percentage,unique_values
0,order_id,str,0,0.00,99441
1,customer_id,str,0,0.00,99441
2,order_status,str,0,0.00,8
3,order_purchase_timestamp,str,0,0.00,98875
4,order_approved_at,str,160,0.16,90733
5,order_delivered_carrier_date,str,1783,1.79,81018
6,order_delivered_customer_date,str,2965,2.98,95664
7,order_estimated_delivery_date,str,0,0.00,459


\nData quality report: order_items
Rows: 112650
Columns: 7
Completely duplicated rows: 0


,column,data_type,missing_values,missing_percentage,unique_values
0,order_id,str,0,0.0,98666
1,order_item_id,int64,0,0.0,21
2,product_id,str,0,0.0,32951
3,seller_id,str,0,0.0,3095
4,shipping_limit_date,str,0,0.0,93318
5,price,float64,0,0.0,5968
6,freight_value,float64,0,0.0,6999


... (4 outputs left out)

\nData quality report: products
Rows: 32951
Columns: 9
Completely duplicated rows: 0


,column,data_type,missing_values,missing_percentage,unique_values
0,product_id,str,0,0.00,32951
1,product_category_name,str,610,1.85,73
2,product_name_lenght,float64,610,1.85,66
3,product_description_lenght,float64,610,1.85,2960
4,product_photos_qty,float64,610,1.85,19
5,product_weight_g,float64,2,0.01,2204
6,product_length_cm,float64,2,0.01,99
7,product_height_cm,float64,2,0.01,102
8,product_width_cm,float64,2,0.01,95


\nData quality report: sellers
Rows: 3095
Columns: 4
Completely duplicated rows: 0


,column,data_type,missing_values,missing_percentage,unique_values
0,seller_id,str,0,0.0,3095
1,seller_zip_code_prefix,int64,0,0.0,2246
2,seller_city,str,0,0.0,611
3,seller_state,str,0,0.0,23


## 4. Review missing values

A missing value is not automatically an error. For example:
- A cancelled or unfinished order may not have a delivery timestamp.
- A customer may submit a review score without writing a comment.
- Product catalogue attributes may be absent in the source.

The next cell prints only columns containing missing values.


In [4]:
for name, df in datasets.items():
    print(f"\\n--- {name.upper()} ---")
    missing = df.isna().sum()
    print(missing[missing > 0].sort_values(ascending=False))

\n--- CUSTOMERS ---
Series([], dtype: int64)
\n--- ORDERS ---
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64
\n--- ORDER_ITEMS ---
Series([], dtype: int64)
\n--- PAYMENTS ---
Series([], dtype: int64)
\n--- REVIEWS ---
review_comment_title      87656
review_comment_message    58247
dtype: int64
\n--- PRODUCTS ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
\n--- SELLERS ---
Series([], dtype: int64)


### 4.1 Investigate order statuses for missing delivery dates

In [5]:
print("All order statuses:")
display(orders["order_status"].value_counts(dropna=False))

print("\\nStatuses among orders with no customer delivery timestamp:")
display(
    orders.loc[
        orders["order_delivered_customer_date"].isna(),
        "order_status"
    ].value_counts(dropna=False)
)

delivered_missing_date = orders.loc[
    orders["order_status"].eq("delivered")
    & orders["order_delivered_customer_date"].isna()
]
print("\\nDelivered orders missing customer delivery timestamp:")
display(delivered_missing_date)

All order statuses:


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

\nStatuses among orders with no customer delivery timestamp:


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

\nDelivered orders missing customer delivery timestamp:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


### 4.2 Investigate product missingness

The previously observed pattern was:
- 610 rows missing the four descriptive fields.
- 2 rows missing physical measurements.
- Row-level missingness can overlap; inspect the number of missing fields per row.

Do not remove these products automatically.


In [6]:
product_missing_columns = products.columns[products.isna().any()].tolist()
print("Missing-value counts:")
display(products[product_missing_columns].isna().sum())

print("\\nNumber of missing fields per product row:")
display(products.isna().sum(axis=1).value_counts().sort_index())

print("\\nSample products with missing values:")
display(products.loc[products.isna().any(axis=1)].head(10))

Missing-value counts:


product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

\nNumber of missing fields per product row:


0    32340
4      610
8        1
Name: count, dtype: int64

\nSample products with missing values:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


### 4.3 Investigate optional review comments

In [7]:
print("Total reviews:", len(reviews))
print("Reviews with title:", reviews["review_comment_title"].notna().sum())
print("Reviews with message:", reviews["review_comment_message"].notna().sum())
print("Reviews without message:", reviews["review_comment_message"].isna().sum())

print("\\nReview score distribution:")
display(reviews["review_score"].value_counts(dropna=False).sort_index())

Total reviews: 99224
Reviews with title: 11568
Reviews with message: 40977
Reviews without message: 58247
\nReview score distribution:


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

## 5. Check completely duplicated rows

These are exact row duplicates, not repeated values in one ID column.


In [8]:
for name, df in datasets.items():
    print(f"{name:15} completely duplicated rows: {df.duplicated().sum():,}")

customers       completely duplicated rows: 0
orders          completely duplicated rows: 0
order_items     completely duplicated rows: 0
payments        completely duplicated rows: 0
reviews         completely duplicated rows: 0
products        completely duplicated rows: 0
sellers         completely duplicated rows: 0


## 6. Validate primary keys

In [9]:
primary_keys = {
    "customers": (customers, "customer_id"),
    "orders": (orders, "order_id"),
    "products": (products, "product_id"),
    "sellers": (sellers, "seller_id")
}

primary_key_results = []

for name, (df, key) in primary_keys.items():
    result = {
        "table": name,
        "key": key,
        "rows": len(df),
        "unique_ids": df[key].nunique(dropna=True),
        "missing_ids": df[key].isna().sum(),
        "duplicate_ids": df[key].duplicated().sum()
    }
    primary_key_results.append(result)

primary_key_report = pd.DataFrame(primary_key_results)
display(primary_key_report)

,table,key,rows,unique_ids,missing_ids,duplicate_ids
0,customers,customer_id,99441,99441,0,0
1,orders,order_id,99441,99441,0,0
2,products,product_id,32951,32951,0,0
3,sellers,seller_id,3095,3095,0,0


## 7. Validate composite keys and repeated IDs

A repeated `order_id` in `order_items` is expected because an order can contain multiple items. The combination `order_id + order_item_id` should identify an item.

A repeated `order_id` in `payments` can also be expected. The combination `order_id + payment_sequential` should identify a payment record.

The Olist reviews file may contain repeated `review_id` values across different orders. Do not drop those rows without understanding the source behavior.


In [10]:
print("ORDER_ITEMS")
print("Repeated order_id:", order_items["order_id"].duplicated().sum())
print(
    "Duplicate order_id + order_item_id:",
    order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
)

print("\\nPAYMENTS")
print("Repeated order_id:", payments["order_id"].duplicated().sum())
print(
    "Duplicate order_id + payment_sequential:",
    payments.duplicated(subset=["order_id", "payment_sequential"]).sum()
)

print("\\nREVIEWS")
print("Repeated review_id:", reviews["review_id"].duplicated().sum())
print("Repeated order_id:", reviews["order_id"].duplicated().sum())
print("Duplicate review rows:", reviews.duplicated().sum())

ORDER_ITEMS
Repeated order_id: 13984
Duplicate order_id + order_item_id: 0
\nPAYMENTS
Repeated order_id: 4446
Duplicate order_id + payment_sequential: 0
\nREVIEWS
Repeated review_id: 814
Repeated order_id: 551
Duplicate review rows: 0


### 7.1 Review duplicate review IDs

In [11]:
duplicate_review_rows = reviews.loc[
    reviews["review_id"].duplicated(keep=False)
].sort_values(["review_id", "order_id"])

review_id_analysis = (
    reviews.groupby("review_id", dropna=False)
    .agg(
        total_records=("order_id", "size"),
        unique_orders=("order_id", "nunique"),
        unique_scores=("review_score", "nunique")
    )
    .query("total_records > 1")
    .sort_values(["total_records", "unique_orders"], ascending=False)
)

print("Duplicate review ID groups:", len(review_id_analysis))
print("Affected rows:", len(duplicate_review_rows))
print("\\nGroup size distribution:")
display(review_id_analysis["total_records"].value_counts().sort_index())
print("\\nSample duplicate review records:")
display(duplicate_review_rows.head(20))
print("\\nSummary of duplicate review ID groups:")
display(review_id_analysis.head(20))

Duplicate review ID groups: 789
Affected rows: 1603
\nGroup size distribution:


total_records
2    764
3     25
Name: count, dtype: int64

\nSample duplicate review records:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


\nSummary of duplicate review ID groups:


,total_records,unique_orders,unique_scores
review_id,,,
08528f70f579f0c830189efc523d2182,3,3,1
0c76e7a547a531e7bf9f0b99cba071c1,3,3,1
1fb4ddc969e6bea80e38deec00393a6f,3,3,1
2172867fd5b1a55f98fe4608e1547b4b,3,3,1
2d6ac45f859465b5c185274a1c929637,3,3,1
308316408775d1600dad81bd3184556d,3,3,1
32415bbf6e341d5d517080a796f79b5c,3,3,1
3415c9f764e478409e8e0660ae816dd2,3,3,1
38821b5c496b678cf91acc34892805ad,3,3,1


**Interpretation:** Repeated `review_id` values are not the same as completely duplicated rows. In the initial inspection, 789 review ID groups were repeated (764 groups of two records and 25 groups of three records). Preserve the records and document the anomaly unless a verified business rule supports a different treatment.

## 8. Create working copies

The raw DataFrames remain unchanged. All transformations from this point are applied to copies.


In [12]:
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()

clean_datasets = {
    "customers": customers_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean,
    "sellers": sellers_clean
}

## 9. Convert timestamp columns to datetime

In [13]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for table_name, columns in date_columns.items():
    df = clean_datasets[table_name]

    for column in columns:
        df[column] = pd.to_datetime(df[column], errors="coerce")

### 9.1 Validate datetime conversion and check newly-created NaT values

In [14]:
original_datasets = {
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews
}

datetime_validation = []

for table_name, columns in date_columns.items():
    for column in columns:
        original = original_datasets[table_name][column]
        converted = clean_datasets[table_name][column]

        newly_invalid = (original.notna() & converted.isna()).sum()

        datetime_validation.append({
            "table": table_name,
            "column": column,
            "dtype_after_conversion": str(converted.dtype),
            "missing_after_conversion": converted.isna().sum(),
            "newly_invalid_conversions": newly_invalid
        })

datetime_validation = pd.DataFrame(datetime_validation)
display(datetime_validation)

,table,column,dtype_after_conversion,missing_after_conversion,newly_invalid_conversions
0,orders,order_purchase_timestamp,datetime64[us],0,0
1,orders,order_approved_at,datetime64[us],160,0
2,orders,order_delivered_carrier_date,datetime64[us],1783,0
3,orders,order_delivered_customer_date,datetime64[us],2965,0
4,orders,order_estimated_delivery_date,datetime64[us],0,0
5,order_items,shipping_limit_date,datetime64[us],0,0
6,reviews,review_creation_date,datetime64[us],0,0
7,reviews,review_answer_timestamp,datetime64[us],0,0


**Expected result from the earlier inspection:** all eight timestamp columns converted successfully, with zero newly invalid conversions. The original missing timestamp counts should remain unchanged.

## 10. Inspect numeric columns with descriptive statistics

In [15]:
numeric_columns = {
    "order_items": ["price", "freight_value", "order_item_id"],
    "payments": [
        "payment_sequential",
        "payment_installments",
        "payment_value"
    ],
    "reviews": ["review_score"],
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
}

for table_name, columns in numeric_columns.items():
    print(f"\\n--- {table_name.upper()} ---")
    display(clean_datasets[table_name][columns].describe().T)

\n--- ORDER_ITEMS ---


,count,mean,std,min,25%,50%,75%,max
price,112650.0,120.653739,183.633928,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.0,19.990320,15.806405,0.00,13.08,16.26,21.15,409.68
order_item_id,112650.0,1.197834,0.705124,1.00,1.00,1.00,1.00,21.00


\n--- PAYMENTS ---


,count,mean,std,min,25%,50%,75%,max
payment_sequential,103886.0,1.092679,0.706584,1.0,1.00,1.0,1.0000,29.00
payment_installments,103886.0,2.853349,2.687051,0.0,1.00,1.0,4.0000,24.00
payment_value,103886.0,154.100380,217.494064,0.0,56.79,100.0,171.8375,13664.08


\n--- REVIEWS ---


,count,mean,std,min,25%,50%,75%,max
review_score,99224.0,4.086421,1.347579,1.0,4.0,5.0,5.0,5.0


\n--- PRODUCTS ---


,count,mean,std,min,25%,50%,75%,max
product_name_lenght,32341.0,48.476949,10.245741,5.0,42.0,51.0,57.0,76.0
product_description_lenght,32341.0,771.495285,635.115225,4.0,339.0,595.0,972.0,3992.0
product_photos_qty,32341.0,2.188986,1.736766,1.0,1.0,1.0,3.0,20.0
product_weight_g,32949.0,2276.472488,4282.038731,0.0,300.0,700.0,1900.0,40425.0
product_length_cm,32949.0,30.815078,16.914458,7.0,18.0,25.0,38.0,105.0
product_height_cm,32949.0,16.937661,13.637554,2.0,8.0,13.0,21.0,105.0
product_width_cm,32949.0,23.196728,12.079047,6.0,15.0,20.0,30.0,118.0


## 11. Count potentially suspicious numeric values

These checks flag values for investigation. A zero is not automatically invalid. For example, a zero freight charge may be a free-shipping promotion. Do not delete or replace values based only on these counts.


In [16]:
numeric_anomaly_counts = {
    "zero_freight_values": (order_items_clean["freight_value"] == 0).sum(),
    "negative_prices": (order_items_clean["price"] < 0).sum(),
    "negative_freight": (order_items_clean["freight_value"] < 0).sum(),
    "zero_payment_values": (payments_clean["payment_value"] == 0).sum(),
    "negative_payment_values": (payments_clean["payment_value"] < 0).sum(),
    "zero_installments": (payments_clean["payment_installments"] == 0).sum(),
    "negative_installments": (payments_clean["payment_installments"] < 0).sum(),
    "zero_product_weights": (products_clean["product_weight_g"] == 0).sum(),
    "negative_product_weights": (products_clean["product_weight_g"] < 0).sum()
}

display(pd.Series(numeric_anomaly_counts, name="count").to_frame())

,count
zero_freight_values,383
negative_prices,0
negative_freight,0
zero_payment_values,9
negative_payment_values,0
zero_installments,2
negative_installments,0
zero_product_weights,4
negative_product_weights,0


### 11.1 Check review score range

In [17]:
invalid_review_scores = reviews_clean.loc[
    ~reviews_clean["review_score"].between(1, 5),
    ["review_id", "order_id", "review_score"]
]

print("Review scores outside 1–5:", len(invalid_review_scores))
display(invalid_review_scores.head(20))

Review scores outside 1–5: 0


,review_id,order_id,review_score


### 11.2 Inspect suspicious records

In [18]:
# Inspect zero or negative payment values
display(
    payments_clean.loc[
        payments_clean["payment_value"] <= 0
    ].head(20)
)

# Inspect zero or negative installments
display(
    payments_clean.loc[
        payments_clean["payment_installments"] <= 0
    ].head(20)
)

# Inspect zero or negative product weights
display(
    products_clean.loc[
        products_clean["product_weight_g"] <= 0
    ].head(20)
)

# Inspect zero freight values
display(
    order_items_clean.loc[
        order_items_clean["freight_value"] == 0
    ].head(20)
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-15 04:31:26,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-14 00:14:29,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-07 05:30:45,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
1784,04105b54650921ca3246f52e6f175f46,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-04-27 09:31:35,69.9,0.0
2232,0517a3e68dac3308995edca2144db36e,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-03 19:11:40,99.9,0.0
2714,061ba2e2d7544790b6ed6b5b4dd9278c,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-09 23:10:46,110.0,0.0
3224,07441f525824bc6b31d4dc19c5d49fc9,1,53b36df67ebb7c41585e8d54d6772e08,4869f7a5dfa277a7dca6462dcf3b52b2,2018-04-26 13:31:30,106.9,0.0
3378,079f16689c29acb6cab92978e6af2137,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-11 18:15:13,69.9,0.0


## 12. Cleaning decisions and rationale

Use this table as the project's cleaning log. These are conservative decisions based on the checks performed so far.

| Issue | Treatment |
|---|---|
| Missing review title/message | Retain as NULL; comments are optional |
| Missing order timestamps | Retain as NULL; may represent incomplete/cancelled orders |
| Delivered status with missing delivery timestamp | Retain and flag for investigation |
| Missing product descriptive attributes | Retain as NULL; investigate before imputing |
| Missing product dimensions/weight | Retain as NULL |
| Product row with all attributes missing | Retain and flag for investigation |
| Repeated order IDs in item/payment tables | Valid one-to-many relationship |
| Repeated review IDs | Retain and document source anomaly |
| Zero/negative numeric values | Investigate; do not change without evidence |
| Timestamp strings | Convert to datetime |
| Completely duplicated rows | No rows were found in the initial inspection |

**No rows are dropped and no numeric values are imputed in this notebook.** If later analysis requires a display label such as `Unknown` for a missing product category, create that label in an analysis-specific column rather than overwriting the source field.


## 13. Create investigation flags (without deleting records)

In [19]:
# Flag delivered orders that have no customer delivery timestamp.
orders_clean["flag_delivered_missing_customer_date"] = (
    orders_clean["order_status"].eq("delivered")
    & orders_clean["order_delivered_customer_date"].isna()
)

# Flag products with missing descriptive attributes.
product_description_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]
products_clean["flag_missing_product_description"] = (
    products_clean[product_description_columns].isna().any(axis=1)
)

# Flag products with missing physical attributes.
product_physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]
products_clean["flag_missing_physical_attributes"] = (
    products_clean[product_physical_columns].isna().any(axis=1)
)

# Flag products with all eight listed attributes missing.
all_product_attributes = product_description_columns + product_physical_columns
products_clean["flag_all_product_attributes_missing"] = (
    products_clean[all_product_attributes].isna().all(axis=1)
)

print("Delivered orders missing customer delivery date:",
      orders_clean["flag_delivered_missing_customer_date"].sum())
print("Products missing at least one descriptive attribute:",
      products_clean["flag_missing_product_description"].sum())
print("Products missing at least one physical attribute:",
      products_clean["flag_missing_physical_attributes"].sum())
print("Products with all eight attributes missing:",
      products_clean["flag_all_product_attributes_missing"].sum())

Delivered orders missing customer delivery date: 8
Products missing at least one descriptive attribute: 610
Products missing at least one physical attribute: 2
Products with all eight attributes missing: 1


## 14. Final validation before saving

In [20]:
# Confirm row counts have not changed.
for name, raw_df in datasets.items():
    clean_df = clean_datasets[name]
    print(
        f"{name:15} raw rows={len(raw_df):>7,} | "
        f"clean rows={len(clean_df):>7,} | "
        f"same row count={len(raw_df) == len(clean_df)}"
    )

# Confirm the expected datetime columns have datetime dtypes.
for table_name, columns in date_columns.items():
    for column in columns:
        assert pd.api.types.is_datetime64_any_dtype(
            clean_datasets[table_name][column]
        ), f"{table_name}.{column} is not datetime"

print("\\nDatetime assertions passed.")

customers       raw rows= 99,441 | clean rows= 99,441 | same row count=True
orders          raw rows= 99,441 | clean rows= 99,441 | same row count=True
order_items     raw rows=112,650 | clean rows=112,650 | same row count=True
payments        raw rows=103,886 | clean rows=103,886 | same row count=True
reviews         raw rows= 99,224 | clean rows= 99,224 | same row count=True
products        raw rows= 32,951 | clean rows= 32,951 | same row count=True
sellers         raw rows=  3,095 | clean rows=  3,095 | same row count=True
\nDatetime assertions passed.


## 15. Save prepared datasets

In [21]:
for name, df in clean_datasets.items():
    output_path = OUTPUT_DIR / f"{name}_cleaned.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path.resolve()}")

Saved: D:\Projects\olist-ecommerce-analysis\data\processed\customers_cleaned.csv
Saved: D:\Projects\olist-ecommerce-analysis\data\processed\orders_cleaned.csv
Saved: D:\Projects\olist-ecommerce-analysis\data\processed\order_items_cleaned.csv
Saved: D:\Projects\olist-ecommerce-analysis\data\processed\payments_cleaned.csv
Saved: D:\Projects\olist-ecommerce-analysis\data\processed\reviews_cleaned.csv
Saved: D:\Projects\olist-ecommerce-analysis\data\processed\products_cleaned.csv
Saved: D:\Projects\olist-ecommerce-analysis\data\processed\sellers_cleaned.csv
